# Entrenar varias CNN en una sola GPU (sin Docker)

Este notebook muestra una forma practica de entrenar **varias CNN en paralelo** sobre la **misma GPU** usando PyTorch.

## Idea
- Cada modelo se entrena en un hilo distinto.
- Cada hilo usa su propio `torch.cuda.Stream()`.
- Se controla la memoria con tamano de batch, precision mixta y numero de modelos concurrentes.

> Si aparece `CUDA out of memory`, baja `BATCH_SIZE` o `NUM_MODELS`.

In [ ]:
import time
from contextlib import nullcontext
from concurrent.futures import ThreadPoolExecutor

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

print('Torch:', torch.__version__)
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True

# Configuracion
NUM_MODELS = 2         # cuantos modelos entrenar a la vez
MODEL_ARCHS = ['small', 'deep']
EPOCHS = 2
BATCH_SIZE = 256
LR = 1e-3
TRAIN_SAMPLES = 12000
VAL_SAMPLES = 2000
IMAGE_SIZE = (1, 28, 28)
NUM_CLASSES = 10

if len(MODEL_ARCHS) != NUM_MODELS:
    raise ValueError('MODEL_ARCHS debe tener exactamente NUM_MODELS elementos.')

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('Este notebook requiere GPU CUDA para entrenamiento concurrente real.')

transform = transforms.Compose([transforms.ToTensor()])

# FakeData evita descargas y hace el ejemplo reproducible en cualquier entorno
train_ds = datasets.FakeData(
    size=TRAIN_SAMPLES,
    image_size=IMAGE_SIZE,
    num_classes=NUM_CLASSES,
    transform=transform,
)
val_ds = datasets.FakeData(
    size=VAL_SAMPLES,
    image_size=IMAGE_SIZE,
    num_classes=NUM_CLASSES,
    transform=transform,
)

def make_loaders():
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=True,
        drop_last=True,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )
    return train_loader, val_loader

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


class DeepCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


def build_model(arch_name, num_classes):
    if arch_name == 'small':
        return SmallCNN(num_classes=num_classes)
    if arch_name == 'deep':
        return DeepCNN(num_classes=num_classes)
    raise ValueError(f'Arquitectura no soportada: {arch_name}')

In [ ]:
def evaluate(model, val_loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            logits = model(xb)
            pred = logits.argmax(dim=1)
            correct += (pred == yb).sum().item()
            total += yb.numel()
    return correct / max(total, 1)


def train_one_model(model_id, arch_name, stream=None):
    device = torch.device('cuda')
    model = build_model(arch_name, num_classes=NUM_CLASSES).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler('cuda')
    train_loader, val_loader = make_loaders()

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            stream_ctx = torch.cuda.stream(stream) if stream is not None else nullcontext()
            with stream_ctx:
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    logits = model(xb)
                    loss = criterion(logits, yb)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            running_loss += loss.item()

        if stream is not None:
            stream.synchronize()

    val_acc = evaluate(model, val_loader, device)
    avg_loss = running_loss / max(len(train_loader), 1)
    return {
        'model_id': model_id,
        'arch': arch_name,
        'train_loss': avg_loss,
        'val_acc': val_acc,
    }

In [ ]:
# Lanzamiento concurrente de varios entrenamientos en la misma GPU
start = time.perf_counter()
streams = [torch.cuda.Stream() for _ in range(NUM_MODELS)]

with ThreadPoolExecutor(max_workers=NUM_MODELS) as executor:
    futures = [
        executor.submit(train_one_model, i, MODEL_ARCHS[i], streams[i])
        for i in range(NUM_MODELS)
    ]
    results = [f.result() for f in futures]

torch.cuda.synchronize()
elapsed = time.perf_counter() - start

print(f'Tiempo total: {elapsed:.2f}s')
for r in sorted(results, key=lambda x: x['model_id']):
    print(
        f"Modelo {r['model_id']} ({r['arch']}): "
        f"loss={r['train_loss']:.4f}, val_acc={r['val_acc']:.4f}"
    )

## Ajustes recomendados para tu caso real

1. Empieza con `NUM_MODELS = 2` y sube gradualmente.
2. Si hay OOM: baja `BATCH_SIZE`, activa/usa FP16 (ya incluido), o reduce `NUM_MODELS`.
3. Monitoriza uso real con `nvidia-smi -l 1` en otra terminal.
4. Para tus datos de audio, cambia `SmallCNN` por tu arquitectura y el `Dataset` por tu loader real.